# 04 — Canny-guided detail Development Evaluation

Evaluate this run's selected detector on its own policy's development (test) split with native FP32 Ultralytics validation, saving per-image matches for dark/bright and Apply/Skip cohorts.

**Inputs:** runs/v4-YYYY-MM-DD/train/weights/best.pt and the run's prepared data.

**Outputs:** runs/v4-YYYY-MM-DD/eval/{metrics.json,paired_stats.npz,plots} only when ALLOW_INFERENCE=True.

**Prerequisites:** Completed 02 and 03. Evaluation never imports or invokes training.

**Outline**

- [1. Configuration](#section-1)
- [2. Policy–Detector Pair](#section-2)
- [3. Evaluation Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

RUN_DATE=None uses the newest run. Saved metrics are reused when inputs match.

In [1]:
import os

import pandas as pd
from IPython.display import display

from vehicle_detection.common import project_root, require_paths
from vehicle_detection.evaluation import evaluate_experiment
from vehicle_detection.experiment_data import load_experiment

ROOT = project_root()
os.chdir(ROOT)  # Run All behaves the same from any kernel working directory.
EXPERIMENT = "v4"
require_paths(
    config=ROOT / f"notebooks/experiments/{EXPERIMENT}/{EXPERIMENT}.yaml",
    shared=ROOT / "configs/data.yaml",
)
config = load_experiment(ROOT, EXPERIMENT)

RUN_DATE = None
ALLOW_INFERENCE = False
display(pd.Series(config["evaluation"], name="evaluation"))
RUN_DATE = '2026-11-10'
ALLOW_INFERENCE = True

imgsz             640
batch              16
device              0
workers             0
conf            0.001
iou               0.7
max_det           300
quantize           32
nms             False
agnostic_nms    False
rect             True
augment         False
plots            True
verbose         False
Name: evaluation, dtype: object

<a id="section-2"></a>

## 2. Policy–Detector Pair

Only development is evaluated; the final holdout is never read.

In [2]:
try:
    result = evaluate_experiment(ROOT, EXPERIMENT, RUN_DATE, allow_inference=ALLOW_INFERENCE)
except FileNotFoundError as error:
    result = None
    print(error)
if result:
    display(pd.DataFrame([result["overall"]]))
    display(pd.DataFrame(result["classes"]).set_index("Class"))

Ultralytics 8.4.198  Python-3.13.9 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)


YOLO26n summary (fused): 122 layers, 2,390,707 parameters, 0 gradients, 5.4 GFLOPs


val: Fast image access  (ping: 0.0±0.0 ms, read: 1413.5±372.1 MB/s, size: 113.7 KB)


val: Scanning D:\truong_research_project\vehicle-detection-branches\main\runs\v4-2026-11-10\data\labels\test... 257 images, 0 backgrounds, 0 corrupt: 1% ──────────── 257/13975 770.9it/s 0.1s<17.8s

val: Scanning D:\truong_research_project\vehicle-detection-branches\main\runs\v4-2026-11-10\data\labels\test... 533 images, 3 backgrounds, 0 corrupt: 3% ──────────── 533/13975 1.4Kit/s 0.2s<9.9s

val: Scanning D:\truong_research_project\vehicle-detection-branches\main\runs\v4-2026-11-10\data\labels\test... 786 images, 3 backgrounds, 0 corrupt: 5% ╸─────────── 786/13975 1.7Kit/s 0.3s<7.7s

... (920 outputs left out)

Speed: 0.3ms preprocess, 3.8ms inference, 0.0ms loss, 0.2ms postprocess per image


Results saved to D:\truong_research_project\vehicle-detection-branches\main\runs\v4-2026-11-10\eval


,precision,recall,mAP50,mAP50_95
0,0.567137,0.394288,0.420345,0.256708


,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP75,mAP50-95
Class,,,,,,,,
car,13820,142541,0.693908,0.675497,0.684579,0.720872,0.421899,0.427682
truck,3773,5965,0.543418,0.467058,0.502353,0.470376,0.360716,0.326579
bus,1759,2276,0.565113,0.405364,0.472091,0.430579,0.356308,0.320282
motorcycle,435,562,0.508740,0.165480,0.249730,0.199016,0.059012,0.086544
bicycle,872,1430,0.524505,0.258042,0.345907,0.280882,0.090950,0.122454


<a id="section-3"></a>

## 3. Evaluation Summary

Cohort AP keeps every GT of the selected images; dark means raw brightness < 0.12.

In [3]:
if result:
    display(
        pd.DataFrame(
            {
                k: {"images": v["images"], "mAP50_95": v["mAP50_95"]}
                for k, v in result["cohorts"].items()
            }
        ).T
    )
    print("Run:", result["inputs"]["run"], "| speed (ms):", result["speed_ms"])
print("Final holdout stays sealed.")

,images,mAP50_95
dark,3307.0,0.226386
bright,10668.0,0.262087
apply,13750.0,0.257090
skip,225.0,0.151030


Run: runs/v4-2026-11-10 | speed (ms): {'preprocess': 0.2516322072315461, 'inference': 3.7507582680290574, 'loss': 0.0004718569315610077, 'postprocess': 0.17837650067637772}
Final holdout stays sealed.
